# 🤖 Redrob AI Ranker — Sandbox Demo

**GitHub:** https://github.com/ashishSoni1234/smart-recruiter-ats

This notebook runs the full 5-stage ranking pipeline on a **small sample of ≤100 candidates**.
It demonstrates the complete architecture end-to-end on CPU without any network calls during ranking.

## Architecture
1. **Stage 1:** BM25 Pre-Filter (100K → 2000)
2. **Stage 2:** Semantic Skill Verification + Honeypot Detection
3. **Stage 3:** Programmatic Disqualifiers
4. **Stage 4:** Hybrid Retrieval (Dense FAISS + Sparse BM25 with Negative Anchor)
5. **Stage 5:** Cross-Encoder Reranking + Candidate-Specific Reasoning

In [ ]:
# ── Step 1: Clone the repo ──────────────────────────────────────────────────
!git clone https://github.com/ashishSoni1234/smart-recruiter-ats.git
%cd smart-recruiter-ats

In [ ]:
# ── Step 2: Install dependencies ────────────────────────────────────────────
!pip install -q orjson pydantic loguru tqdm sentence-transformers faiss-cpu rank-bm25 python-dotenv groq
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('✅ Dependencies installed')

In [ ]:
# ── Step 3: Use the sample candidates (≤100 for sandbox) ───────────────────
import json, os

# Load sample_candidates.json (included in repo)
with open('data/sample_candidates.json', 'r') as f:
    sample_data = json.load(f)

print(f'Sample candidates loaded: {len(sample_data)}')

# Write as JSONL for the pipeline
os.makedirs('data', exist_ok=True)
with open('data/sample_candidates.jsonl', 'w') as f:
    for cand in sample_data:
        f.write(json.dumps(cand) + '\n')

print('✅ Sample JSONL written')

In [ ]:
# ── Step 4: Run the pipeline ─────────────────────────────────────────────────
# Note: The JD intent is pre-cached in data/jd_intent_cache.json
# No network calls are made during ranking — fully offline.

!python pipeline.py \
    --candidates data/sample_candidates.jsonl \
    --jd data/jd_extracted.txt \
    --output sandbox_output.csv \
    --format jsonl

In [ ]:
# ── Step 5: View results ────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv('sandbox_output.csv')
print(f'Total rows: {len(df)}')
print('\nTop 10 candidates:')
pd.set_option('display.max_colwidth', 120)
display(df.head(10)[['candidate_id', 'rank', 'score', 'reasoning']])

In [ ]:
# ── Step 6: Validate submission format ──────────────────────────────────────
!python data/validate_submission.py sandbox_output.csv

# Show score distribution
import matplotlib.pyplot as plt

df = pd.read_csv('sandbox_output.csv')
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(df['rank'], df['score'], color='steelblue', alpha=0.8)
plt.xlabel('Rank'); plt.ylabel('Score'); plt.title('Score by Rank (non-increasing ✅)')
plt.subplot(1, 2, 2)
plt.hist(df['score'], bins=20, color='coral', alpha=0.8)
plt.xlabel('Score'); plt.ylabel('Count'); plt.title('Score Distribution')
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=100)
plt.show()
print('✅ Sandbox run complete!')